# Handling Errors

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Catch an exception with try and catch, and keep the program running.
- Throw an exception when a method gets a value it can't accept.
- Create your own exception class.

**In this notebook**, one idea at a time (run each cell before moving on):
1. What an exception is
2. try and catch: handling it
3. The usual suspects
4. Several catch blocks, and finally
5. throw: refusing a bad value
6. Exceptions travel up the call stack
7. Your own exception class

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## 1. What an exception is

When something goes wrong while the program runs (a division by zero, an index past the end, text that isn't a number), Java **throws an exception**: it stops what it was doing and, unless some code handles it, the whole program stops with a message. That message tells you the **kind** of exception, a **detail**, and the **line** where it happened. Reading it is the first step to fix the bug.

In [ ]:
int[] scores = {7, 9, 4};
int index = 2;
IO.println("Before");
IO.println(scores[index]);
IO.println("After");

🧪 **Try it:** change `index` to `3` and run: find the exception's kind, the detail (which index, what length) and the line. Is "After" printed?

## 2. try and catch: handling it

Put the risky code in a `try` block. If it throws, Java jumps straight to the matching `catch` block, and the program **keeps running** after it. If nothing is thrown, the `catch` is skipped. `e.getMessage()` gives the detail.

```java
try {
    // code that may throw
} catch (NumberFormatException e) {
    // runs only if that kind was thrown
}
```

In [ ]:
String[] typed = {"12", "abc", "7"};
for (String text : typed) {
    try {
        int n = Integer.parseInt(text);
        IO.println(text + " doubled is " + n * 2);
    } catch (NumberFormatException e) {
        IO.println("\"" + text + "\" isn't a number: " + e.getMessage());
    }
}
IO.println("The program goes on");

🧪 **Try it:** add `"4.5"` and `""` to the array. Then move the `try` outside the loop: why does the program now stop at the first bad value?

## 3. The usual suspects

| Exception | Usually means |
|---|---|
| `NumberFormatException` | `parseInt` / `parseDouble` got text that isn't a number |
| `ArithmeticException` | a whole-number division by zero |
| `ArrayIndexOutOfBoundsException` | an index below 0 or at least `length` |
| `NullPointerException` | using a variable that holds `null`, as if it pointed to an object |
| `IllegalArgumentException` | a method got a value it can't accept |

In [ ]:
try {
    IO.println(10 / 0);
} catch (ArithmeticException e) {
    IO.println("ArithmeticException: " + e.getMessage());
}
try {
    String name = null;
    IO.println(name.length());
} catch (NullPointerException e) {
    IO.println("NullPointerException: the variable held null");
}
IO.println(10 / 0.0);   // with a double there's no exception: Infinity

🧪 **Try it:** change the first `catch` to `NumberFormatException`: the exception no longer matches, so nobody catches it and the cell stops. Put it back.

### ✏️ Exercise: A safe index

Return values[i], or -1 when i is outside the array. Catch the ArrayIndexOutOfBoundsException instead of checking first.

In [ ]:
int safeGet(int[] values, int i) {
    // TODO: read values[i] inside try
    return -1;
}

void main() {
    check("inside", safeGet(new int[] {4, 5}, 1), 5);
    check("too far", safeGet(new int[] {4, 5}, 2), -1);
    check("negative", safeGet(new int[] {4, 5}, -1), -1);
}

<details>
<summary>Show an answer</summary>

```java
int safeGet(int[] values, int i) {
    try {
        return values[i];
    } catch (ArrayIndexOutOfBoundsException e) {
        return -1;
    }
}

void main() {
    check("inside", safeGet(new int[] {4, 5}, 1), 5);
    check("too far", safeGet(new int[] {4, 5}, 2), -1);
    check("negative", safeGet(new int[] {4, 5}, -1), -1);
}
```

</details>

## 4. Several catch blocks, and finally

A `try` can have several `catch` blocks, one per kind; Java uses the first that matches, so put the **more specific** kinds first. A `finally` block runs **always**, whether an exception happened or not, which suits clean-up work such as closing a file.

In [ ]:
int[] values = {10, 0, 5};
String[] picks = {"0", "1", "7", "x"};
for (String pick : picks) {
    try {
        int i = Integer.parseInt(pick);
        IO.println("100 / values[" + i + "] = " + 100 / values[i]);
    } catch (NumberFormatException e) {
        IO.println(pick + ": not a number");
    } catch (ArithmeticException e) {
        IO.println(pick + ": division by zero");
    } catch (ArrayIndexOutOfBoundsException e) {
        IO.println(pick + ": no such index");
    } finally {
        IO.println("  (done with " + pick + ")");
    }
}

🧪 **Try it:** add a first `catch (IllegalArgumentException e)` above the `NumberFormatException` one and read the error: `NumberFormatException` is a kind of `IllegalArgumentException`, so the second catch could never run.

## 5. throw: refusing a bad value

Your own methods can throw too. When a method gets a value it can't work with, `throw new IllegalArgumentException("why")` stops it right there, instead of returning a wrong answer. Whoever called it decides what to do, with a `try`.

In [ ]:
double average(int[] values) {
    if (values.length == 0) {
        throw new IllegalArgumentException("no values to average");
    }
    int total = 0;
    for (int v : values) {
        total += v;
    }
    return (double) total / values.length;
}

void main() {
    IO.println(average(new int[] {7, 8, 9}));
    try {
        IO.println(average(new int[] {}));
    } catch (IllegalArgumentException e) {
        IO.println("Couldn't compute: " + e.getMessage());
    }
}

🧪 **Try it:** remove the `if` with the `throw` and run: what does the average of nothing give now? Which is better, an exception or a strange number?

### ✏️ Exercise: Throw on bad input

Return the age when it is 0 or more; otherwise throw `new IllegalArgumentException("negative age")`. The test catches it and reads its message.

In [ ]:
/** Calls checkedAge and returns the exception's message, or "no exception". */
String messageFor(int age) {
    try {
        checkedAge(age);
        return "no exception";
    } catch (IllegalArgumentException e) {
        return e.getMessage();
    }
}

int checkedAge(int age) {
    // TODO: throw when age < 0
    return age;
}

void main() {
    check("a valid age", checkedAge(30), 30);
    check("zero", checkedAge(0), 0);
    check("negative", messageFor(-1), "negative age");
}

<details>
<summary>Show an answer</summary>

```java
/** Calls checkedAge and returns the exception's message, or "no exception". */
String messageFor(int age) {
    try {
        checkedAge(age);
        return "no exception";
    } catch (IllegalArgumentException e) {
        return e.getMessage();
    }
}

int checkedAge(int age) {
    if (age < 0) {
        throw new IllegalArgumentException("negative age");
    }
    return age;
}

void main() {
    check("a valid age", checkedAge(30), 30);
    check("zero", checkedAge(0), 0);
    check("negative", messageFor(-1), "negative age");
}
```

</details>

## 6. Exceptions travel up the call stack

An exception that isn't caught where it's thrown leaves that method and goes to the one that called it, then to its caller, and so on, until some `catch` handles it, or it leaves `main` and stops the program. The slides follow one exception up the call stack.

#### 🎞️ An exception climbing the call stack

![parseInt throws a NumberFormatException; it leaves every method without a catch until main catches it](exception-steps.svg)

In [ ]:
int age(String text) {
    IO.println("  age(\"" + text + "\") starts");
    int value = Integer.parseInt(text);     // may throw here
    IO.println("  age ends");               // skipped when it throws
    return value;
}

void main() {
    for (String text : new String[] {"30", "x"}) {
        try {
            IO.println("Age: " + age(text));
        } catch (NumberFormatException e) {
            IO.println("main caught it: " + e.getMessage());
        }
    }
}

🧪 **Try it:** move the `try` / `catch` into `age`, returning -1 when it fails: now main never sees the exception. Which place makes more sense?

## 7. Your own exception class

An exception is an object of a class, so you can create your own for problems specific to your program. Extend `RuntimeException` and callers don't have to catch it. Extend `Exception` and it becomes **checked**: every method that may throw it must say `throws` in its header, and the compiler makes callers deal with it.

In [ ]:
class EmptyNameException extends Exception {
    EmptyNameException() {
        super("the name is empty");
    }
}

String greeting(String name) throws EmptyNameException {
    if (name.isBlank()) {
        throw new EmptyNameException();
    }
    return "Hi, " + name.trim() + "!";
}

void main() {
    for (String name : new String[] {"Ada", "   "}) {
        try {
            IO.println(greeting(name));
        } catch (EmptyNameException e) {
            IO.println("Error: " + e.getMessage());
        }
    }
}

🧪 **Try it:** remove the `try` / `catch` in `main` and read the error: a checked exception must be caught or declared. Then change `extends Exception` to `extends RuntimeException`: does it compile now?

## In short

When something goes wrong, Java **throws** an exception, and the program stops unless some code **catches** it. Put the risky code in `try`; a `catch` block runs only when its kind of exception is thrown.

```java
int age(String text) {
    int value = Integer.parseInt(text.trim());
    if (value < 0 || value > 150) {
        throw new IllegalArgumentException("Not a real age: " + value);
    }
    return value;
}

void main() {
    String line = IO.readln();
    try {
        IO.println("Next year: " + (age(line) + 1));
    } catch (NumberFormatException e) {
        IO.println("Not a number: " + line);
    } catch (IllegalArgumentException e) {
        IO.println(e.getMessage());
    }
}
```

Your own exception is a class that `extends Exception` (callers must catch it or declare `throws`) or `extends RuntimeException` (they don't have to).

**Watch out:** put the more specific `catch` first. `NumberFormatException` is a kind of `IllegalArgumentException`, so in the other order the compiler refuses the code.

## Readings

- [Catching and Handling Exceptions](https://dev.java/learn/language/annotations-exceptions/exceptions/catching-handling/) (dev.java, 15 min): try and catch blocks, catching more than one exception, and finally. The main example writes to a file; if it feels heavy, the page suggests jumping to the common patterns at its end. Try-with-resources is more than you need now.
- [Throwing Exceptions](https://dev.java/learn/language/annotations-exceptions/exceptions/throwing/) (dev.java, 8 min): Optional, for Insufficient Funds: the throw statement, the throws clause, and creating your own exception class. Chained exceptions are more than you need now.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Safe Division (try / catch)** · Easy
- **Parse Numbers (NumberFormatException)** · Medium
- **Ask Until Valid** · Medium
- **Insufficient Funds (Custom Exception)** · Medium
- **Robust Calculator** · Hard